# Exploration: GB-IE electricity system (Solution)

<img src="https://docs.pypsa.org/latest/assets/logo/logo-primary-light.svg#only-light" width="300px" />

:::{note}
Also in this tutorial, you might want to refer to the PyPSA documentation: https://docs.pypsa.org.
:::


:::{note}
If you have not yet set up Python on your computer, you can execute this tutorial in your browser via [Google Colab](https://colab.research.google.com/). Download the `.ipynb` file using the download button on the top right corner and import it in [Google Colab](https://colab.research.google.com/).

Then install the following packages by executing the following command in a Jupyter cell at the top of the notebook.

```sh
!pip install pypsa pandas matplotlib plotly pydeck geopandas
```
:::

First things first! We need a few packages for this tutorial:

In [ ]:
import geopandas as gpd
import pypsa
import pandas as pd
import pydeck as pdk
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import plotly.io as pio
import plotly.offline as py

pd.options.plotting.backend = "plotly"

In [ ]:
pypsa.options.params.optimize.include_objective_constant = True

## Importing the regional case study

The following electricity-only PyPSA network was generated using the European model PyPSA-Eur (https://pypsa-eur.readthedocs.io) and has the following properties:

- Regions: Great-Britain, Ireland
- Resolution:
    - Regional: NUTS2
    - Temporal: hourly
- Transmission line expansion: off
- Load shedding costs: 3000 €/MW

In [ ]:
n = pypsa.Network(
    "https://tubcloud.tu-berlin.de/s/4tdQNNEDT38xAqt/download/network-gb-ie.nc"
)
n.sanitize()  # ensuring that all carriers have a color

Let's also import regional shapes that surround the administrative boundaries of the electricity buses.

In [ ]:
regions = gpd.read_file(
    "https://tubcloud.tu-berlin.de/s/HEHKZYckdnkeNjB/download/regions-gb-ie.geojson"
)

## Exercises

### Task 1: Before solving

Before solving the model, let's have a look into the imported PyPSA network and learn about its properties. 

#### 1.a. What time periods or snapshots does the network cover?

In [ ]:
print(f"Number of snapshots: {len(n.snapshots)}\n")
print(f"Snapshots:")
print(n.snapshots)
print("\nThe snapshots are hourly timestamps covering the first week of the year 2024.")

#### 1.b. Based on the properties of the network, can you tell whether this is a dispatch or capacity expansion model?

In [ ]:
# To do this, we check the `_extendable` attribute for each component in the network.
print(f"Extendable generators: {n.generators.p_nom_extendable.unique()}")
print(f"Extendable lines: {n.lines.s_nom_extendable.unique()}")
print(f"Extendable links: {n.links.p_nom_extendable.unique()}")
print(f"Extendable storage units: {n.storage_units.p_nom_extendable.unique()}")

# No stores
# print(f"Extendable stores: {n.stores.e_nom_extendable.unique()}")

print(
    "\nNone of the generation, storage, AC and DC transmission line capacities are extendable.\nSolving this network is equivalent to solving a standard nodal dispatch problem."
)

#### 1.c. What is the total existing HVDC interconnector capacity in the network?

In [ ]:
dc_interconnector_cap = n.links.p_nom.sum()
print(f"Total DC interconnector capacity: {dc_interconnector_cap:.1f} MW")

#### 1.d. In which region of do we observe the highest peak load? Can you find it on a map?

In [ ]:
max_nodal_loads = n.loads_t.p_set.max(axis=0)
bus_id = max_nodal_loads.idxmax()
print(f"Bus with the highest load: {bus_id}")

In [ ]:
# Map peak loads to bus sizes
bus_size = max_nodal_loads
n.explore(bus_size=bus_size, auto_scale=True, bus_alpha=0.7)

### Task 2: Solve the model

In the next step, we solve the network using the open-source solver HiGHS.

In [ ]:
n.optimize(solver_name="highs")

#### 2.a. What are the total system costs?

In [ ]:
objective = n.objective / 1e6
# Alternative
# objective = n.statistics.opex().sum()/1e6
print(f"Total system cost: {objective:.2f} million EUR")

In [ ]:
# Alternative
# objective = n.statistics.opex().sum()/1e6
print(f"Total system cost: {objective:.2f} million EUR")

#### 2.b. Plot the hourly dispatch.

In [ ]:
n.statistics.energy_balance.iplot.area(bus_carrier="AC")

### Task 3: Visualisation on a map

#### 3.a. Store the nodal energy balances in a `pd.Series`, indexed over `bus` and `carrier`.

Note: Set `nice_names` to `False` for correct mapping.

In [ ]:
nodal_energy_balances = n.statistics.energy_balance(
    components=["Generator", "Load", "StorageUnit"],
    aggregate_across_components=True,
    groupby=["bus", "carrier"],
    nice_names=False,
)
print(nodal_energy_balances.head())

#### 3.b. Store the total transmission line and link flows in the variables `line_flow` and `link_flow`

In [ ]:
line_flow = n.lines_t.p0.sum(axis=0)
link_flow = n.links_t.p0.sum(axis=0)

#### 3.c. Now stitch all the components together and visualise them in `n.explore()`

In [ ]:
# optional settings
view_state = {}
view_state["zoom"] = 5
view_state["pitch"] = 35  # Up/down angle relative to the map's plane

map = n.explore(
    view_state=view_state,
    map_style="dark",
    bus_size=nodal_energy_balances,  # MWh -> km²
    bus_split_circle=True,
    bus_size_max=7000,  # km²
    line_color="yellow",
    line_width=line_flow,  # MWh -> km
    link_width=link_flow,  # MWh -> km
    line_flow=line_flow,  # MWh -> km
    link_flow=link_flow,  # MWh -> km
    branch_width_max=16,  # km
    auto_scale=True,
    bus_columns=["v_nom"],
    line_columns=["s_nom"],
    link_columns=["p_nom"],
    arrow_size_factor=2,
    tooltip=True,  # disabled here for technical limits of mkdocs-jupyter plugin
)

In [ ]:
map

#### 3.d. What are the demand-weighted average nodal prices?

In [ ]:
nodal_prices = n.statistics.prices()  # by default this is demand weighted
print(nodal_prices.head())

#### 3.e. Bonus: Add the the prices as a background layer to the interactive map in 3.c.

- For stacking layers, please refer to pydeck's documentation: https://deckgl.readthedocs.io/en/latest/layer.html
- You can also find an example in PyPSA's documentation: https://docs.pypsa.org/latest/user-guide/plotting/explore/

In [ ]:
# Map average prices by shapeISO
regions["avg_price"] = regions["name"].map(nodal_prices).round(2)

In [ ]:
values = regions["avg_price"]
cmap = plt.get_cmap("Greens")
norm = mcolors.Normalize(vmin=values.min(), vmax=values.max())

In [ ]:
def price_to_color(price, alpha=0.7):
    color = cmap(norm(price))  # RGBA in 0-1
    rgb = [round(c * 255) for c in color[:3]]  # only RGB
    a = round(alpha * 255)
    return rgb + [a]

In [ ]:
regions["color"] = regions["avg_price"].apply(price_to_color)
regions.head()

In [ ]:
# Add a custom tooltip column (HTML or plain text)
regions["tooltip_html"] = (
    "<b>Region:</b> "
    + regions["name"]
    + "<br>"
    + "<b>NUTS:</b> "
    + regions["name"]
    + "<br><b>Avg. Price:</b> "
    + regions["avg_price"].astype(str)
    + " €/MWh"
)

# Create layer
regions_layer = pdk.Layer(
    "GeoJsonLayer",
    regions,
    stroked=True,
    filled=True,
    get_fill_color="color",
    get_line_color=[255, 255, 255, 255],
    line_width_min_pixels=1,
    pickable=True,
    auto_highlight=True,
)

In [ ]:
map.layers.insert(0, regions_layer)

In [ ]:
map